# Webpage UI Element Classification
## Multi-label Classification from Full-Page Screenshot Images

The objective of this project is the classification of UI element types from full webpage screenshot images retrieved from the synthetically generated Roboflow Website Screenshots dataset [Website Screenshots](https://public.roboflow.com/object-detection/website-screenshots), composed of 1206 unique screenshots from over 1000 of the world's top websites. 


This leads to a supervised multi-label classification with 8 output binary labels per image defined in the datasets specs:

| Label | Description |
|---|---|
| `button` | Navigation links, tabs, interactive controls |
| `field` | Form input fields |
| `heading` | Text enclosed in `<h1>`–`<h6>` tags |
| `iframe` | Ads and third-party embedded content |
| `image` | `<img>`, `<svg>`, `<video>` tags and icons |
| `label` | Text labelling form fields |
| `link` | Inline textual `<a>` tags |
| `text` | All other text content |


Each screenshot can have multiple active labels simultaneously.

## 1. Motivation

The understanding of which UI elements are visually present in a webpage has direct applications in different circumstances and domains. Specifically, the ability to predict UI elements from pixels alone is relevant for the following examples:

- **Robotic Process Automation (RPA):** RPA tools are designed to *"capture the execution of routines previously performed by a human user on the interface of a computer system, and then emulate their enactment in place of the user"* [Agostinelli et al., 2019]. Identifying which UI elements are visually present (e.g. buttons, fields, links, etc.) is a prerequisite for any agent operating without HTML access. 

- **Accessibility auditing:** *"the automated system does not have access to any meta-data about the user interface, such as view hierarchies or accessibility tags, or if this information is missing or incompletely defined, as is often the case."* [Wu et al., 2021]. This means that many applications simply don't provide enough of these metadata, making this kind of detection a good alternative for WCAG (Web Content Accessibility Guidelines) compliance checking.

- **Web information extraction:** Extracting structured data from web pages is challenging due to *"the unstructured nature of textual data and the diverse layout patterns of the web documents."* [Wang et al., 2022]. Classifying which UI element types are visually present is a key first step in any extraction pipeline.


### References
- Wu, J., Zhang, X., Nichols, J., & Bigham, J. P. (2021). Screen Parsing: Towards Reverse Engineering of UI Models from Screenshots. *UIST '21*. https://doi.org/10.1145/3472749.3474763
- Agostinelli, S., Marrella, A., & Mecella, M. (2019). Research Challenges for Intelligent Robotic Process Automation. *BPM 2019 Workshops*. https://doi.org/10.1007/978-3-030-37453-2_2
- Wang, Q., et al. (2022). WebFormer: The Web-page Transformer for Structure Information Extraction. *WWW '22*. https://doi.org/10.1145/3485447.3512032

## 2. Set Up

In [3]:
# Import libraries
import os
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing import image as keras_image
from sklearn.decomposition import PCA, NMF
from sklearn.manifold import MDS
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import normalize
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import squareform
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.multiclass import OneVsRestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
from sklearn.metrics import (hamming_loss, f1_score, roc_auc_score,
                             precision_recall_curve, average_precision_score)
from tqdm import tqdm


plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')

print(f"TensorFlow : {tf.__version__}")
print(f"NumPy      : {np.__version__}")

TensorFlow : 2.16.2
NumPy      : 1.26.4


In [19]:
# Configuration of global constants
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Folders path
DATA_DIR = 'data/raw'
EMB_DIR  = 'embeddings'
FIG_DIR  = 'figures'

# Create folders if not done yet
os.makedirs(EMB_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

# 8 UI element types
CLASSES    = ['button', 'field', 'heading', 'iframe', 'image', 'label', 'link', 'text']

# For ResNet50, it expects images of 224x224 pixels
IMG_SIZE   = (224, 224)

# Process images in groups of 32
BATCH_SIZE = 32

print("Directories:")
print(f"  Data       : {DATA_DIR}")
print(f"  Embeddings : {EMB_DIR}")
print(f"  Figures    : {FIG_DIR}")
print(f"\nClasses ({len(CLASSES)}): {CLASSES}")
print(f"Image size  : {IMG_SIZE}")
print(f"Batch size  : {BATCH_SIZE}")
print(f"Random seed : {RANDOM_SEED}")

Directories:
  Data       : data/raw
  Embeddings : embeddings
  Figures    : figures

Classes (8): ['button', 'field', 'heading', 'iframe', 'image', 'label', 'link', 'text']
Image size  : (224, 224)
Batch size  : 32
Random seed : 42


## 3. Data Loading and Preparation

Load and deduplicate (1206 images)
Class distribution bar chart
Label co-occurrence heatmap
Cardinality histogram (how many labels per image)
Sample image grid with labels overlaid

In [24]:
# Load and deduplicate: when downloaded, Roboflow export shows 2412 images rather than 1206
# Both copies of images have identical pixels and labels, so it doesn't make sense to keep both.

dfs = {}
for split in ['train', 'valid', 'test']:
    df = pd.read_csv(os.path.join(DATA_DIR, split, '_classes.csv'))
    df.columns = df.columns.str.strip()
    df['domain']   = df['filename'].apply(lambda x: x.split('_png')[0])
    df['split']    = split
    df             = df.drop_duplicates(subset='domain').reset_index(drop=True)
    df['filepath'] = df['filename'].apply(
        lambda x: os.path.join(DATA_DIR, split, x)
    )
    dfs[split] = df
    print(f"{split:6s}: {len(df):4d} images")

train_df = dfs['train']
valid_df = dfs['valid']
test_df  = dfs['test']
all_df   = pd.concat(dfs.values(), ignore_index=True)

print(f"\nTotal : {len(all_df)} unique images")
print(f"Labels: {CLASSES}")

FileNotFoundError: [Errno 2] No such file or directory: 'data/raw/train/_classes.csv'

## 3. Data Preparation

Resize images to 224×224
Normalize with ImageNet stats
Handle class imbalance (class weights)
Extract frozen ResNet50 embeddings → save to embeddings/

## 4. Exploratory Data Analysis

PCA on embeddings
MDS
K-means clustering
NMF on label matrix
Outlier detection (Isolation Forest, LOF)
Hierarchical clustering

## 5. Model Building & Tuning

OneVsRest wrapper
Logistic Regression baseline
SVM, Random Forest, XGBoost
Cross-validated hyperparameter tuning

## 6. Model Evaluation

Hamming loss, micro/macro F1, per-class ROC-AUC
Precision-recall curves

## 7. Interpretation & Insights

Saliency maps
Feature importance per class
Failure case analysis

## 8. Final Discussion

Limitations
Proposed improvements